# Electrical and electro-optic PCells

gpdk ships a small set of hand-written devices next to the generic photonic library. They share the photonic SOI silicon and the M1-M3 BEOL:

| Cell | Source PDK | Ports |
|---|---|---|
| `nch`, `pch` | gf180mcu `nfet` / `pfet` (3.3V) | `Drain_i`, `Source_i`, `Gate_i`, `Bulk_i` |
| `cmim` | gf180mcu `cap_mim` (MIM-A, M2/M3) | `top`, `bottom` |
| `cmom` | IHP SG13G2 `cmom` | `PLUS`, `MINUS` |
| `res_poly` | gf180mcu `ppolyf`/`npolyf` `_u`/`_s` | `r0`, `r1` |
| `res_metal` | gf180mcu `rm1`-`rm3` | `r0`, `r1` |
| `straight_pn_ribbed` | ribbed PN phase shifter | `o1`, `o2`, `anode`, `cathode` |

In [ ]:
import gdsfactory as gf

import gpdk
from gpdk import PDK
from gpdk.layer_stack import LAYER_STACK

PDK.activate()


def show(c: gf.Component) -> None:
    """Plot a cell and print its ports and info."""
    c.plot()
    print(c.name, dict(c.info))
    for p in c.ports:
        print(f"  {p.name:10s} {p.port_type:10s} {p.layer} {p.center}")

## MOS transistors

`nf` sets the number of gate fingers. `grw=0` removes the guard ring and its bulk contacts.

In [ ]:
show(gpdk.cells.nch())
show(gpdk.cells.nch(w_gate=1.0, l_gate=0.5, nf=4))
show(gpdk.cells.pch(nf=2))
show(gpdk.cells.pch(nf=2, grw=0))

## Capacitors

In [ ]:
for lc, wc in [(5, 5), (20, 10)]:
    show(gpdk.cells.cmim(lc=lc, wc=wc))

In [ ]:
show(gpdk.cells.cmom())
show(gpdk.cells.cmom(nfingers=5, length=8, botmetal="M1", topmetal="M2"))

## Resistors

`_u` variants are unsilicided and draw a silicide block (SAB); `_s` variants are silicided.

In [ ]:
for res_type in ["ppolyf_u", "npolyf_u", "ppolyf_s", "npolyf_s"]:
    c = gpdk.cells.res_poly(l_res=4, w_res=1, res_type=res_type)
    print(res_type, f"{c.info['resistance_ohm']:.1f} ohm")
show(gpdk.cells.res_poly(l_res=4, w_res=1, res_type="ppolyf_u"))

In [ ]:
for metal in ["M1", "M2", "M3"]:
    show(gpdk.cells.res_metal(l_res=10, w_res=0.5, metal=metal))

## Ribbed PN phase shifter

`length` snaps down to a whole number of rib periods. The lateral PN junction sits at `junction_offset` above the waveguide axis.

In [ ]:
show(gpdk.cells.straight_pn_ribbed(length=50))

## Composing devices

The devices are plain gdsfactory cells, so they compose and route like any other gpdk cell.

## Layer stack

New levels: `active`, `poly`, `mim`, and one level per implant (`n`, `p`, `np`, `pp`, `npp`, `ppp`, `nwell`) carrying its ion and concentration in `info`.

In [ ]:
new = ["active", "poly", "mim", "n", "p", "np", "pp", "npp", "ppp", "nwell"]
for name in new:
    level = LAYER_STACK.layers[name]
    print(
        f"{name:7s} zmin={level.zmin:5.2f} thickness={level.thickness:5.2f} "
        f"{level.material:4s} {level.info}"
    )